In [21]:
# Đoạn code này tự động lấy Tên tài khoản và Tên máy tính
import getpass
import socket
user = getpass.getuser()
host = socket.gethostname()
def get_system_context():
    user = getpass.getuser()
    host = socket.gethostname()
    return f"[USER: {user}] [HOST: {host}]"

In [22]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
listings_df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .load("../data/Listings.csv")
    # .load("hdfs://master-node:9000/AirbnbData/Listings.csv")  # dùng khi master-node bật
listings_df_raw.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import ArrayType, StringType
listings_df = listings_df_raw.withColumn(
    "amenities",
    from_json(col("amenities"), ArrayType(StringType()))
)

In [ ]:
from pyspark.sql.functions import when, col
# Quy đổi toàn bộ giá từ tiền tệ địa phương (Local Currency) sang USD
# Tỷ giá được cập nhật theo thời điểm hiện tại
listings_df_currency = listings_df.withColumn(
    "price_usd",
    when(col("city") == "New York", col("price") * 1.0)           # USD giữ nguyên
    .when(col("city") == "Paris", col("price") * 1.156)            # EUR -> USD
    .when(col("city") == "Rome", col("price") * 1.156)             # EUR -> USD
    .when(col("city") == "Sydney", col("price") * 0.7038)           # AUD -> USD
    .when(col("city") == "Hong Kong", col("price") * 0.1276)        # HKD -> USD
    .when(col("city") == "Bangkok", col("price") * 0.03048)         # THB -> USD
    .when(col("city") == "Mexico City", col("price") * 0.05795)     # MXN -> USD
    .when(col("city") == "Cape Town", col("price") * 0.06130)       # ZAR -> USD
    .when(col("city") == "Istanbul", col("price") *  0.02162)        # TRY -> USD
    .when(col("city") == "Rio de Janeiro", col("price") * 0.1961 )   # BRL -> USD
    .otherwise(col("price"))
)

In [ ]:
listings_df_currency.show()

In [ ]:
listings_df_currency.printSchema()

In [ ]:
reviews_df=spark.read.csv("../data/Reviews.csv",header=True,inferSchema=True)
# reviews_df=spark.read.csv("hdfs://master-node:9000/AirbnbData/Reviews.csv",header=True,inferSchema=True)  # dùng khi master-node bật
reviews_df.show()

In [ ]:
reviews_df.printSchema()

In [ ]:
listings_df_currency.createOrReplaceTempView("listings")
reviews_df.createOrReplaceTempView("reviews")
spark.catalog.listTables()

In [ ]:
# Minh chứng số dòng > 100.000 dòng
spark.sql("SELECT COUNT(*) AS Total_rows FROM listings").show()

In [ ]:
# Minh chứng số cột >10
num_cols = len(listings_df.columns)
print(f"Tổng số cột: {num_cols}")

In [ ]:
# Câu truy vấn 2: Đánh giá hiệu quả của danh hiệu Superhost đối với việc thu hút khách hàng và chất lượng dịch vụ
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   CASE L.host_is_superhost
       WHEN 't' THEN 'superhost'
       ELSE 'none'
   END AS host_is_superhost,
   COUNT(DISTINCT L.listing_id) AS total_listings,
   COUNT(R.review_id) AS reviews,
   ROUND(AVG(L.host_response_rate)*100, 2) AS avg_host_reponse,
   ROUND(AVG(L.host_acceptance_rate)*100, 2) AS avg_host_acceptance,
   ROUND(AVG(L.review_scores_rating), 2) AS avg_rating,
   ROUND(AVG(L.review_scores_cleanliness), 2) AS avg_cleanliness,
   ROUND(AVG(L.review_scores_communication), 2) AS avg_communication,
   ROUND(AVG(L.review_scores_value), 2) AS avg_value
FROM listings L
JOIN reviews R ON L.listing_id = R.listing_id
WHERE L.host_is_superhost IS NOT NULL
GROUP BY L.host_is_superhost
ORDER BY L.host_is_superhost ASC;""").toPandas()

In [ ]:
# Câu truy vấn 3: Cơ cấu nguồn cung phòng và giá thuê theo quy mô sức chứa khách tại từng thành phố
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   city,
   CASE
       WHEN accommodates <= 2 THEN '1-2 khách (Cá nhân/Cặp đôi)'
       WHEN accommodates BETWEEN 3 AND 5 THEN '3-5 khách (Gia đình/Nhóm nhỏ)'
       ELSE 'Trên 5 khách (Đoàn du lịch)'
   END AS room_capacity_segment,
   COUNT(listing_id) AS total_rooms_listed,
   ROUND(AVG(price_usd), 2) AS avg_nightly_price
FROM listings
WHERE accommodates IS NOT NULL
GROUP BY city,
   CASE
       WHEN accommodates <= 2 THEN '1-2 khách (Cá nhân/Cặp đôi)'
       WHEN accommodates BETWEEN 3 AND 5 THEN '3-5 khách (Gia đình/Nhóm nhỏ)'
       ELSE 'Trên 5 khách (Đoàn du lịch)'
   END
ORDER BY city,room_capacity_segment ASC;""").toPandas()

In [ ]:
# Câu truy vấn 7: Đánh giá các tiện ích cốt lõi thuộc phân khúc căn hộ Cao cấp
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   exploded_amenity AS amenity_name,
   COUNT(*) AS total_occurrence
FROM listings
LATERAL VIEW EXPLODE(amenities) AS exploded_amenity
WHERE price_usd > (SELECT AVG(price_usd) FROM listings)
GROUP BY exploded_amenity
ORDER BY total_occurrence DESC
LIMIT 5;""").toPandas()

In [ ]:
# QUERY 1 – Diem hanh vi tot & ty le Superhost
spark.sql("""
  SELECT diem_hanh_vi_tot,
         COUNT(*)                       AS so_listing,
         ROUND(100.0*AVG(la_super), 2)  AS ty_le_superhost
  FROM (
      SELECT CASE WHEN host_is_superhost = 't' THEN 1 ELSE 0 END AS la_super,
             ( CASE WHEN host_response_time   = 'within an hour' THEN 1 ELSE 0 END
             + CASE WHEN host_identity_verified = 't'            THEN 1 ELSE 0 END
             + CASE WHEN instant_bookable       = 't'            THEN 1 ELSE 0 END ) AS diem_hanh_vi_tot
      FROM listings
  )
  GROUP BY diem_hanh_vi_tot
  ORDER BY diem_hanh_vi_tot
""").toPandas()

In [ ]:
# QUERY 9 – Top 5 Superhost rui ro nhat trong tung nhom
spark.sql("""
WITH host_agg AS (
 SELECT host_id, FIRST(city) AS city, COUNT(*) AS n_listing,
        ROUND(AVG(review_scores_rating),2)  AS rating_tb,
        MAX(host_is_superhost) AS is_super
 FROM listings
 WHERE review_scores_rating IS NOT NULL
 GROUP BY host_id
 HAVING MAX(host_is_superhost) = 't'
),
city_bench AS (
 SELECT city, AVG(rating_tb) AS rating_city FROM host_agg GROUP BY city
),
RankedAnalysis AS (
 SELECT h.host_id, h.city, h.n_listing, h.rating_tb,
        ROUND(h.rating_tb - c.rating_city, 2) AS lech_vs_city,
        NTILE(4) OVER (ORDER BY h.rating_tb) AS nhom_rui_ro,
        CASE WHEN h.rating_tb < c.rating_city THEN 'SUPERHOST RUI RO'
             ELSE 'OK' END AS canh_bao
 FROM host_agg h JOIN city_bench c ON h.city = c.city
 WHERE h.n_listing >= 2
)
SELECT host_id, city, n_listing, rating_tb, lech_vs_city, nhom_rui_ro, canh_bao
FROM (
 SELECT *,
        ROW_NUMBER() OVER (PARTITION BY nhom_rui_ro ORDER BY lech_vs_city ASC) AS thut_tu_trong_nhom
 FROM RankedAnalysis
)
WHERE thut_tu_trong_nhom <= 5
ORDER BY nhom_rui_ro ASC, lech_vs_city ASC
""").toPandas(20)

In [ ]:
# QUERY 10 – PIVOT: Ty le Superhost theo loai phong & thanh pho
spark.sql("""
SELECT * FROM (
 SELECT city, room_type,
        CASE WHEN host_is_superhost='t' THEN 1.0 ELSE 0.0 END AS is_super
 FROM listings
) PIVOT (
 ROUND(AVG(is_super)*100, 1)
 FOR room_type IN ('Entire place' AS entire, 'Private room' AS private,
                   'Hotel room' AS hotel, 'Shared room' AS shared)
)
""").toPandas()
